# Web Scraping — Agência Minas Gerais

Neste notebook vamos coletar notícias da editoria de **Esportes** do portal [Agência Minas Gerais](https://www.agenciaminas.mg.gov.br), o serviço oficial de comunicação do governo do Estado de Minas Gerais.

O objetivo é construir uma base de dados estruturada com título, descrição, data, temas e texto completo de cada notícia publicada entre 2019 e 2026, para alimentar as análises de PLN e aprendizado de máquina dos próximos notebooks.

## Objetivo

Queremos extrair automaticamente as notícias de esportes do portal Agência Minas — sem precisar acessar cada página manualmente.

A URL de entrada do nosso alvo é:

```
https://www.agenciaminas.mg.gov.br/noticias?page=1&tema=esportes&territorio=
```

Cada página lista 10 notícias. O portal tem 33 páginas com conteúdo de esportes, totalizando **323 notícias** a serem coletadas.

## Por que o portal Agência Minas é um bom alvo?

Nossa principal ferramenta nesse momento é o **"Ver Código Fonte"** do navegador (`Ctrl+U` ou clique direito → "Ver código fonte da página").

### Sites difíceis de raspar
- **Renderização via JavaScript** — o conteúdo só aparece depois que scripts executam no navegador (ex: portais de notícias como G1, UOL). Para esses casos seria necessário usar o **Selenium**, que controla um navegador de verdade.
- **Paginação via "ver mais"** — o conteúdo é carregado dinamicamente por requisições AJAX.

### Por que o portal Agência Minas é fácil
- O HTML já vem **completamente renderizado** na resposta da requisição — não há JavaScript bloqueando o conteúdo.
- A **paginação é via URL editável**: basta trocar `page=1` por `page=2`, `page=3`, etc.
- A estrutura HTML é **consistente e bem organizada**, com IDs e classes semânticas que facilitam a extração.

**Encontramos nosso alvo!**

## As ferramentas da coleta

#### **`requests`**
- Biblioteca utilizada para **fazer requisições HTTP**.
- Simula um navegador acessando a URL e captura o HTML bruto da resposta.
- Usamos um `User-Agent` real no cabeçalho para evitar que o servidor bloqueie a requisição como um bot.

#### **`BeautifulSoup`** (do pacote `bs4`)
- Biblioteca usada para **analisar e navegar pela estrutura HTML**.
- Permite localizar elementos pela tag, classe CSS ou atributo — como `find('main', id='blocknews')` para achar o bloco de notícias.
- Facilita a extração de textos, links e atributos de qualquer elemento da página.

### Fluxo básico:
1. `requests.get(url)` — faz a requisição e captura o HTML.
2. `BeautifulSoup(html, 'html.parser')` — transforma o HTML em um objeto navegável.
3. `soup.find(...)` / `soup.find_all(...)` — localiza os elementos com os dados que queremos.

## Capturando o HTML da página de listagem

A imagem abaixo mostra a página de notícias de esportes do portal Agência Minas. Cada card de notícia que vemos visualmente corresponde a um bloco `<section class="row padding-y-2">` no HTML — é nesse bloco que estão o link, os temas e a data de cada notícia da listagem.

In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
# URL da página de listagem de notícias de esportes
url = "https://www.agenciaminas.mg.gov.br/noticias?page=1&tema=esportes&territorio="

# Cabeçalho simulando um navegador Chrome real para evitar bloqueio
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [3]:
response = requests.get(url, headers=headers)
response
# <Response [200]> -> requisição bem-sucedida
# <Response [403]> -> servidor bloqueou a requisição (sem User-Agent correto)

<Response [200]>

In [4]:
response.text

'<!DOCTYPE html>\n<html dir="ltr" xml:lang="pt-br" lang="pt-br" xmlns="http://www.w3.org/1999/xhtml" xmlns:fb="http://www.facebook.com/2008/fbml" itemscope itemtype="http://schema.org/">\n<head>\n  <link rel="shortcut icon" type="image/x-icon" href="/assets/favicon-ff67004ef63143778f63114d3e89a222ac5ece29230ef719706855c99f8b02f4.ico" />\n  <link rel="stylesheet" media="all" href="/assets/application-cba39f7486a5bedb9e50b106fb2d54e0693a506c6ac4582bde60e61797f5da1b.css" data-turbolinks-track="true" />\n  <title>Agência Minas Gerais | Notícias</title>\n<meta name="description" content="Acompanhe todas as notícias oficiais do Governo de Minas Gerais relacionadas aos temas de interesse da população." />\n<meta name="keywords" content="agência minas gerais, governo estadual, estado, notícias, agropecuária, ciência e tecnologia, cultura, defesa social, desenvolvimento agrário, desenvolvimento econômico, direitos humanos, educação, esportes, fazenda, gestão, governo, infraestrutura, meio ambie

In [5]:
soup = BeautifulSoup(response.text, "html.parser")

In [6]:
soup

<!DOCTYPE html>

<html dir="ltr" itemscope="" itemtype="http://schema.org/" lang="pt-br" xml:lang="pt-br" xmlns="http://www.w3.org/1999/xhtml" xmlns:fb="http://www.facebook.com/2008/fbml">
<head>
<link href="/assets/favicon-ff67004ef63143778f63114d3e89a222ac5ece29230ef719706855c99f8b02f4.ico" rel="shortcut icon" type="image/x-icon"/>
<link data-turbolinks-track="true" href="/assets/application-cba39f7486a5bedb9e50b106fb2d54e0693a506c6ac4582bde60e61797f5da1b.css" media="all" rel="stylesheet"/>
<title>Agência Minas Gerais | Notícias</title>
<meta content="Acompanhe todas as notícias oficiais do Governo de Minas Gerais relacionadas aos temas de interesse da população." name="description"/>
<meta content="agência minas gerais, governo estadual, estado, notícias, agropecuária, ciência e tecnologia, cultura, defesa social, desenvolvimento agrário, desenvolvimento econômico, direitos humanos, educação, esportes, fazenda, gestão, governo, infraestrutura, meio ambiente, saúde, social, trabalho 

In [7]:
# Exibe o HTML como texto puro (sem tags)
print(soup.get_text())






Agência Minas Gerais | Notícias












































Agência Minas Gerais



Abrir e fechar menu






fonte
A
A

contraste









buscar










fonte
A
A
contraste



Agência Minas Gerais




notícias

Últimas Notícias
Arquivo: notícias de 2005-2015
Arquivo: notícias de 2015-2018


multimídia

Áudios
Vídeos
Fotos


programe-se

governador

Inicial
Agenda oficial
Áudios
Vídeos
Fotos


Sala de Imprensa
Sites do Governo
Serviços
contato
















buscar





Últimas Notícias


Acessar filtro de busca


Filtrar por:



Governador

Governador






 Escolha o tema
TemasAcordo do Rio Doce
Agropecuária
Bate-Papo com o Governador
Brumadinho
Ciência e Tecnologia
Coronavírus
Cultura
Defesa Civil
Desenvolvimento Econômico
Dia da Mulher 
Direitos Humanos
Educação
Ensino Superior
Esportes
Fazenda
Gestão
Governador
Governo
Infraestrutura
Meio Ambiente
Minas Consciente
Pesquisa
Previdência
Publicidade
Recupera Minas
Reforma da Previdência
Saúde
Segurança
Servi

## Encontrando o padrão estrutural

Ao inspecionar o código-fonte do portal Agência Minas (Ctrl+U no navegador), identificamos que todas as notícias da listagem estão dentro de um contêiner principal:

```html
<main id="blocknews">
```

Dentro desse `<main>`, cada notícia é representada por uma `<section>` com as classes `row` e `padding-y-2`:

```html
<section class="row / padding-y-2">
    <a class="link-container" href="/noticia/governo-de-minas-inaugura-quadra-poliesportiva-em-andradas-e-fortalece-infraestrutura-escolar">
        <!-- imagem da notícia -->
    </a>
    <div class="col-xxs col-xs-7 col-md-9">
        <!-- temas (btn-tema), data e título -->
        <a class="btn-tema bg-governo" href="/noticias?tema=governo">Governo</a>
        <a class="btn-tema bg-esportes" href="/noticias?tema=esportes">Esportes</a>
        <a class="btn-tema bg-educacao" href="/noticias?tema=educacao">Educação</a>
    </div>
</section>
```

O padrão que precisamos para extrair os links das notícias é:
- Encontrar `<main id="blocknews">`
- Dentro dele, encontrar todas as tags `<a>` cujo `href` começa com `/noticia/`
- Concatenar com a URL base: `https://www.agenciaminas.mg.gov.br` + `href`

In [8]:
# Localizando o bloco principal de notícias
bloco_noticias = soup.find("main", id="blocknews")
bloco_noticias

<main id="blocknews">
<section class="row / padding-y-2">
<a class="link-container" href="/noticia/governo-de-minas-participa-de-premiacao-em-etapa-do-jemg-no-sul-do-estado">
<div class="col-auto col-xs-5 col-md-3 / margin-xs-bottom-1 margin-sm-bottom-0">
<div class="image-container unveil">
<img alt="Unveil 8a21a6f614db065bb1ca2c21a5c803aede783f93e5277febbd47f37d9dbae4ca" class="img-responsive" data-src="/system/news/images/000/130/096/thumb/Foto_Jemg_2.jpg?1779633209" src="/assets/unveil-8a21a6f614db065bb1ca2c21a5c803aede783f93e5277febbd47f37d9dbae4ca.gif"/>
</div>
</div>
</a>
<div class="col-xxs col-xs-7 col-md-9">
<div class="margin-bottom-1 / hidden-xs">
<a class="btn-tema bg-social / margin-right-1" href="/noticias?tema=social">Social</a>
<a class="btn-tema bg-esportes / margin-right-1" href="/noticias?tema=esportes">Esportes</a>
<a class="btn-tema bg-governador / margin-right-1" href="/noticias?tema=governador">Governador</a>
</div>
<a class="link-container" href="/noticia/gover

In [9]:
# Cada notícia é uma <section> com a classe 'padding-y-2'
secoes = bloco_noticias.find_all("section", class_=lambda c: c and "padding-y-2" in c)
secoes

[<section class="row / padding-y-2">
 <a class="link-container" href="/noticia/governo-de-minas-participa-de-premiacao-em-etapa-do-jemg-no-sul-do-estado">
 <div class="col-auto col-xs-5 col-md-3 / margin-xs-bottom-1 margin-sm-bottom-0">
 <div class="image-container unveil">
 <img alt="Unveil 8a21a6f614db065bb1ca2c21a5c803aede783f93e5277febbd47f37d9dbae4ca" class="img-responsive" data-src="/system/news/images/000/130/096/thumb/Foto_Jemg_2.jpg?1779633209" src="/assets/unveil-8a21a6f614db065bb1ca2c21a5c803aede783f93e5277febbd47f37d9dbae4ca.gif"/>
 </div>
 </div>
 </a>
 <div class="col-xxs col-xs-7 col-md-9">
 <div class="margin-bottom-1 / hidden-xs">
 <a class="btn-tema bg-social / margin-right-1" href="/noticias?tema=social">Social</a>
 <a class="btn-tema bg-esportes / margin-right-1" href="/noticias?tema=esportes">Esportes</a>
 <a class="btn-tema bg-governador / margin-right-1" href="/noticias?tema=governador">Governador</a>
 </div>
 <a class="link-container" href="/noticia/governo-de-m

In [10]:
print(f"{len(secoes)} notícias encontradas na página 1\n")

10 notícias encontradas na página 1



**Por que usamos `<section class='padding-y-2'>` como marcador?**

Essa classe CSS é usada exclusivamente para separar visualmente cada notícia na listagem. Ela funciona como um "separador de registros" — cada `<section>` com essa classe equivale a uma notícia. Identificamos isso ao inspecionar várias páginas do portal e verificar que o padrão era consistente em todas elas.

In [11]:
# "/noticia/estudantes-da-rede-estadual-conquistam-o-1-lugar-no-campeonato-mundial-de-jiu-jitsu"

In [12]:
# Extraindo os links de notícias da página (sem duplicatas e sem links de tema)
# Os links de notícias sempre começam com '/noticia/' — isso os diferencia
# dos links de categoria como '/noticias?tema=esportes'
BASE = "https://www.agenciaminas.mg.gov.br"

links = list(dict.fromkeys(
    BASE + a['href'] for a in bloco_noticias.find_all('a', href=True)
    if a['href'].startswith('/noticia/')
))

# Exibindo os links extraídos
for link in links:
    print(link)

https://www.agenciaminas.mg.gov.br/noticia/governo-de-minas-participa-de-premiacao-em-etapa-do-jemg-no-sul-do-estado
https://www.agenciaminas.mg.gov.br/noticia/governo-de-minas-inaugura-nova-quadra-poliesportiva-em-escola-estadual-de-teofilo-otoni
https://www.agenciaminas.mg.gov.br/noticia/minas-gerais-acelera-rumo-a-copa-do-mundo-feminina-2027-com-foco-em-legado-social-e-esportivo
https://www.agenciaminas.mg.gov.br/noticia/governo-de-minas-inaugura-quadra-poliesportiva-em-andradas-e-fortalece-infraestrutura-escolar
https://www.agenciaminas.mg.gov.br/noticia/governo-de-minas-vistoria-obras-de-revitalizacao-de-ginasio-poliesportivo-em-pocos-de-caldas
https://www.agenciaminas.mg.gov.br/noticia/governo-de-minas-vistoria-instalacoes-do-programa-minas-urbano-em-ipatinga
https://www.agenciaminas.mg.gov.br/noticia/igarape-inaugura-segundo-centro-esportivo-com-recursos-do-acordo-de-brumadinho
https://www.agenciaminas.mg.gov.br/noticia/governo-do-estado-lanca-3-campeonato-mineiro-de-futebol-das

**O que significa `'a', href=True`?**

- **`'a'`**: Refere-se à tag `<a>` em HTML, que é usada para criar hiperlinks.
- **`href=True`**: Filtra apenas as tags `<a>` que possuem o atributo `href` definido (não vazios).
- **`startswith('/noticia/')`**: Filtra somente links que levam a uma notícia individual, descartando links de navegação, temas e outros.
- **`dict.fromkeys(...)`**: Remove duplicatas mantendo a ordem original — uma notícia pode aparecer com dois links iguais na mesma `<section>` (um na imagem e outro no título).

## Acessando cada notícia individualmente

Temos os links das notícias da listagem. Agora precisamos acessar cada URL individualmente e extrair os campos de interesse.

Ao inspecionar o HTML de uma notícia no portal Agência Minas, identificamos que as informações estruturadas ficam em **meta tags** no `<head>` da página — um padrão amplamente usado por portais de notícias para compartilhamento em redes sociais (Open Graph Protocol):

```html
<meta property="og:title" content="Título da notícia" />
<meta name="description" property="og:description" content="Resumo..." />
<meta property="og:image" content="https://.../imagem.jpg" />
<meta property="article:published_time" content="2026-03-30T15:10:00-03:00" />
```

Os **temas** ficam em botões com a classe `btn-tema` no corpo da página, e o **texto completo** fica dentro da tag `<article class="clear">`.

In [13]:
# Acessando a primeira notícia da lista
full_url = links[0]
response = requests.get(full_url, headers=headers)

print(full_url)

https://www.agenciaminas.mg.gov.br/noticia/governo-de-minas-participa-de-premiacao-em-etapa-do-jemg-no-sul-do-estado


In [14]:
soup = BeautifulSoup(response.text, "html.parser")

In [15]:
soup

<!DOCTYPE html>

<html dir="ltr" itemscope="" itemtype="http://schema.org/" lang="pt-br" xml:lang="pt-br" xmlns="http://www.w3.org/1999/xhtml" xmlns:fb="http://www.facebook.com/2008/fbml">
<head>
<link href="/assets/favicon-ff67004ef63143778f63114d3e89a222ac5ece29230ef719706855c99f8b02f4.ico" rel="shortcut icon" type="image/x-icon"/>
<link data-turbolinks-track="true" href="/assets/application-cba39f7486a5bedb9e50b106fb2d54e0693a506c6ac4582bde60e61797f5da1b.css" media="all" rel="stylesheet"/>
<title>Agência Minas Gerais | Governo de Minas participa de premiação em etapa do Jemg no Sul do estado</title>
<meta content="  O governador de Minas Gerais, Mateus Simões, participou, nesse sábado (23/5), da cerimônia de encerramento e premiação da etapa..." name="description"/>
<meta content="governo, de, minas, participa, de, premiação, em, etapa, do, jemg, no, sul, do, estado" name="keywords"/>
<meta content="Governo de Minas participa de premiação em etapa do Jemg no Sul do estado" property=

In [16]:
print(soup.get_text(separator="\n").strip())

Agência Minas Gerais | Governo de Minas participa de premiação em etapa do Jemg no Sul do estado


























































































Agência Minas Gerais








Abrir e fechar menu














fonte


A


A




contraste




















buscar






















fonte


A


A


contraste








Agência Minas Gerais










notícias




Últimas Notícias


Arquivo: notícias de 2005-2015


Arquivo: notícias de 2015-2018






multimídia




Áudios


Vídeos


Fotos






programe-se




governador




Inicial


Agenda oficial


Áudios


Vídeos


Fotos






Sala de Imprensa


Sites do Governo


Serviços


contato


































buscar


















Social


Esportes


Governador








download



















Dom 24 maio 2026




11:20




atualizado em Dom 24 maio 2026 12:04




Governo de Minas participa de premiação em etapa do Jemg no Sul do estado


Jogos Escolares de Minas Gerais voltaram 

### Mapeamento completo dos seletores

A página de cada notícia do portal Agência Minas segue uma estrutura HTML consistente. Mapeamos os seguintes seletores após inspecionar diversas notícias:

```html
<!-- TÍTULO: meta tag Open Graph -->
<meta property="og:title" content="Atletas de MG vencem Jogos Escolares Nacionais"/>

<!-- DESCRIÇÃO: meta tag padrão -->
<meta name="description" content="O programa Revelando Atletas apoiou..."/>

<!-- DATA DE PUBLICAÇÃO: meta tag article -->
<meta property="article:published_time" content="2024-03-15T14:22:00-03:00"/>

<!-- TEMAS: botões btn-tema no corpo da página -->
<a class="btn-tema bg-governo"  href="/noticias?tema=governo">Governo</a>
<a class="btn-tema bg-esportes" href="/noticias?tema=esportes">Esportes</a>
<a class="btn-tema bg-educacao" href="/noticias?tema=educacao">Educação</a>

<!-- SUBTÍTULO: primeira tag h2 do artigo -->
<h2>Competição reuniu mais de 5.000 estudantes de todo o país</h2>

<!-- TEXTO COMPLETO: artigo principal -->
<article class="clear">
    <p>Texto da notícia...</p>
</article>
```

In [17]:
# Extraindo o título via meta og:title
title = soup.find("meta", property="og:title")["content"]

# Extraindo a descrição via meta name='description'
description = soup.find("meta", attrs={"name": "description"})["content"]

# Extraindo a data de publicação via tag <time datetime=...>
time_tag = soup.find("time", datetime=True)
date_str = time_tag["datetime"] if time_tag else None

# Extraindo os temas (botões btn-tema), sem duplicatas
# Cada notícia pode ter múltiplos temas (ex: Esportes, Governo, Social)
temas = list(dict.fromkeys(
    a.get_text(strip=True)
    for a in soup.find_all("a", class_=lambda c: c and "btn-tema" in c)
))

# Extraindo a imagem via meta og:image
img = soup.find("meta", property="og:image")["content"]

# Extraindo o texto completo da notícia (<article class="clear">)
# Removemos tags de script, style e figure para ficar só com o texto
article = soup.find("article", class_="clear")
if article:
    for tag in article.find_all(["script", "style", "figure"]):
        tag.decompose()
    texto = article.get_text(separator="\n").strip()

print("Título:", title)
print("Descrição:", description)
print("Temas:", temas)
print("Data:", date_str)
print("Imagem:", img)
print("Texto (primeiras 200 chars):", texto[:200])

Título: Governo de Minas participa de premiação em etapa do Jemg no Sul do estado
Descrição:   O governador de Minas Gerais, Mateus Simões, participou, nesse sábado (23/5), da cerimônia de encerramento e premiação da etapa...
Temas: ['Social', 'Esportes', 'Governador']
Data: 2026-05-24 11:20:00 -0300
Imagem: https://agenciaminas.mg.gov.br/system/news/images/000/130/096/original/Foto_Jemg_2.jpg?1779633209
Texto (primeiras 200 chars): Dom 24 maio 2026




11:20




atualizado em Dom 24 maio 2026 12:04




Governo de Minas participa de premiação em etapa do Jemg no Sul do estado


Jogos Escolares de Minas Gerais voltaram a registrar


In [18]:
informacoes = {
    "url": full_url,
    "titulo": title,
    "descricao": description,
    "temas": temas,
    "imagem": img,
    "data": date_str,
    "texto": texto,
}

informacoes

{'url': 'https://www.agenciaminas.mg.gov.br/noticia/governo-de-minas-participa-de-premiacao-em-etapa-do-jemg-no-sul-do-estado',
 'titulo': 'Governo de Minas participa de premiação em etapa do Jemg no Sul do estado',
 'descricao': '\xa0 O governador de Minas Gerais, Mateus Simões, participou, nesse sábado (23/5), da cerimônia de encerramento e premiação da etapa...',
 'temas': ['Social', 'Esportes', 'Governador'],
 'imagem': 'https://agenciaminas.mg.gov.br/system/news/images/000/130/096/original/Foto_Jemg_2.jpg?1779633209',
 'data': '2026-05-24 11:20:00 -0300',
 'texto': 'Dom 24 maio 2026\n\n\n\n\n11:20\n\n\n\n\natualizado em Dom 24 maio 2026 12:04\n\n\n\n\nGoverno de Minas participa de premiação em etapa do Jemg no Sul do estado\n\n\nJogos Escolares de Minas Gerais voltaram a registrar adesão recorde neste ano, com a participação dos 853 municípios mineiros\n\n\n\n\n\n\n\n\n\n\n\n\nGil Leonardi / Imprensa MG\n\n\n\n\n\n\n\n\n\n\n\n\n\n\ndownload da imagem\n\n\n\ue2c4\n\n\n\n\n\n\n\n\n\

## Coletando todas as páginas

Agora que validamos a extração em uma única notícia, vamos automatizar a coleta para todas as 33 páginas do portal.

A estratégia é:
1. Iterar de `page=1` até `page=33`
2. Em cada página, localizar o `<main id="blocknews">` e extrair todos os links `/noticia/...`
3. Usar `time.sleep()` com delay randômico entre 0.3 e 0.8 segundos para respeitar o servidor e evitar bloqueio por rate limiting

In [19]:
import time
import random

In [20]:
# Detecta automaticamente o número total de páginas
response_pg1 = requests.get(
    "https://www.agenciaminas.mg.gov.br/noticias?page=1&tema=esportes",
    headers=headers
)
soup_pg1 = BeautifulSoup(response_pg1.text, "html.parser")

paginacao = soup_pg1.find("div", class_="pagination")
ultima_pagina = max(
    int(a.get_text()) for a in paginacao.find_all("a")
    if a.get_text().strip().isdigit()
)
print(f"Total de páginas encontradas: {ultima_pagina}")

Total de páginas encontradas: 33


In [21]:
numero_paginas = 33

links_noticias = []

for p in range(1, numero_paginas + 1):
    url = f"https://www.agenciaminas.mg.gov.br/noticias?page={p}&tema=esportes"
    response = requests.get(url, headers=headers)

    print(p, response)

    soup_p = BeautifulSoup(response.text, "html.parser")
    bloco = soup_p.find("main", id="blocknews")

    if bloco:
        novos_links = list(dict.fromkeys(
            BASE + a["href"] for a in bloco.find_all("a", href=True)
            if a["href"].startswith("/noticia/")
        ))
        links_noticias += novos_links

    # Delay randômico para não sobrecarregar o servidor
    tempo_espera = random.uniform(0.3, 0.8)
    time.sleep(tempo_espera)

print(f"\nTotal de links coletados: {len(links_noticias)}")

1 <Response [200]>
2 <Response [200]>
3 <Response [200]>
4 <Response [200]>
5 <Response [200]>
6 <Response [200]>
7 <Response [200]>
8 <Response [200]>
9 <Response [200]>
10 <Response [200]>
11 <Response [200]>
12 <Response [200]>
13 <Response [200]>
14 <Response [200]>
15 <Response [200]>
16 <Response [200]>
17 <Response [200]>
18 <Response [200]>
19 <Response [200]>
20 <Response [200]>
21 <Response [200]>
22 <Response [200]>
23 <Response [200]>
24 <Response [200]>
25 <Response [200]>
26 <Response [200]>
27 <Response [200]>
28 <Response [200]>
29 <Response [200]>
30 <Response [200]>
31 <Response [200]>
32 <Response [200]>
33 <Response [200]>

Total de links coletados: 326


In [22]:
len(links_noticias)

326

## Salvando as notícias individualmente

Com todos os links em mãos, acessamos cada notícia individualmente e extraímos todos os campos mapeados anteriormente. Cada notícia é salva como um arquivo `.json` separado na pasta `data_minas/`, para facilitar a reprocessamento sem precisar raspar novamente.

In [ ]:
import json
import pathlib

data_dir = pathlib.Path("data_minas")
data_dir.mkdir(exist_ok=True)

for i, full_url in enumerate(links_noticias):
    response = requests.get(full_url, headers=headers)
    soup_n = BeautifulSoup(response.text, "html.parser")

    title = soup_n.find("meta", property="og:title")
    title = title["content"] if title else None

    description = soup_n.find("meta", attrs={"name": "description"})
    description = description["content"] if description else None

    date = soup_n.find("meta", property="article:published_time")
    date = date["content"] if date else None

    temas = list(dict.fromkeys(
        a.get_text(strip=True)
        for a in soup_n.find_all("a", class_=lambda c: c and "btn-tema" in c)
    ))

    img = soup_n.find("meta", property="og:image")
    imagem = img["content"] if img else None

    texto_div = soup_n.find("div", id="content")
    texto = texto_div.get_text(separator="\n").strip() if texto_div else ""

    informacoes = {
        "url": full_url,
        "titulo": title,
        "descricao": description,
        "temas": temas,
        "imagem": imagem,
        "data": date,
        "texto": texto,
    }

    file_path = data_dir / f"noticia_{i}.json"
    with file_path.open("w", encoding="utf-8") as f:
        json.dump(informacoes, f, ensure_ascii=False, indent=4)

    print(f"[{i+1}/{len(links_noticias)}] Salvo: {file_path.name}")

    tempo_espera = random.uniform(0.2, 0.6)
    time.sleep(tempo_espera)

[1/326] Salvo: noticia_0.json
[2/326] Salvo: noticia_1.json
[3/326] Salvo: noticia_2.json
[4/326] Salvo: noticia_3.json
[5/326] Salvo: noticia_4.json
[6/326] Salvo: noticia_5.json
[7/326] Salvo: noticia_6.json
[8/326] Salvo: noticia_7.json
[9/326] Salvo: noticia_8.json
[10/326] Salvo: noticia_9.json
[11/326] Salvo: noticia_10.json
[12/326] Salvo: noticia_11.json
[13/326] Salvo: noticia_12.json
[14/326] Salvo: noticia_13.json
[15/326] Salvo: noticia_14.json
[16/326] Salvo: noticia_15.json
[17/326] Salvo: noticia_16.json
[18/326] Salvo: noticia_17.json
[19/326] Salvo: noticia_18.json
[20/326] Salvo: noticia_19.json
[21/326] Salvo: noticia_20.json
[22/326] Salvo: noticia_21.json
[23/326] Salvo: noticia_22.json
[24/326] Salvo: noticia_23.json
[25/326] Salvo: noticia_24.json
[26/326] Salvo: noticia_25.json
[27/326] Salvo: noticia_26.json
[28/326] Salvo: noticia_27.json
[29/326] Salvo: noticia_28.json
[30/326] Salvo: noticia_29.json
[31/326] Salvo: noticia_30.json
[32/326] Salvo: noticia_31.

## Coleta paralela com joblib

A coleta sequencial demora muito: com ~0.4s de delay por notícia e 323 notícias, seriam mais de 2 minutos.

Usamos **`joblib.Parallel`** com `prefer='threads'` para processar múltiplas notícias simultaneamente. Com 8 threads, o tempo de coleta cai para uma fração. Isso é possível porque o gargalo é a espera pela resposta do servidor (I/O), não o processamento da CPU.

Nesta versão mais completa também extraímos campos adicionais:
- **`subtitulo`**: primeira tag `<h2>` do artigo
- **`fotografo` e `orgao_foto`**: crédito da foto, parseado da div `news-image-text`
- **`n_relacionadas`**: número de notícias relacionadas linkadas na página
- **`n_palavras_titulo`**: contagem de palavras do título

In [ ]:
from joblib import Parallel, delayed

def parsear_credito(credito):
    if not credito or credito.strip() == "":
        return None, None
    credito = credito.replace("download da imagem", "").strip()
    if "/" in credito:
        partes = credito.split("/", 1)
        return partes[0].strip(), partes[1].strip()
    return None, credito.strip()

def processar_noticia(i, full_url):
    response = requests.get(full_url, headers=headers)
    soup_n = BeautifulSoup(response.text, "html.parser")

    # Título via meta og:title
    title = soup_n.find("meta", property="og:title")
    title = title["content"] if title else None

    # Descrição via meta name='description'
    description = soup_n.find("meta", attrs={"name": "description"})
    description = description["content"] if description else None

    # Data via tag <time datetime=...>
    time_tag = soup_n.find("time", datetime=True)
    date_str = time_tag["datetime"] if time_tag else None

    # Hora de atualização (texto visível do <time>)
    hora_atualizacao = time_tag.get_text(separator=" ", strip=True) if time_tag else ""

    # Temas via botões btn-tema
    temas = list(dict.fromkeys(
        a.get_text(strip=True)
        for a in soup_n.find_all("a", class_=lambda c: c and "btn-tema" in c)
    ))

    # Imagem via meta og:image
    img = soup_n.find("meta", property="og:image")
    imagem = img["content"] if img else None

    # Crédito da foto via div news-image-text
    credito_tag = soup_n.find("div", class_=lambda c: c and "news-image-text" in str(c))
    credito_foto = credito_tag.get_text(strip=True) if credito_tag else ""

    # Texto completo via <article class="clear">
    article = soup_n.find("article", class_="clear")
    if article:
        for tag in article.find_all(["script", "style", "figure"]):
            tag.decompose()
        texto = article.get_text(separator="\n").strip()
    else:
        texto = ""

    # Subtítulo via primeira tag <h2>
    subtitulo_tag = soup_n.find("h2")
    subtitulo = subtitulo_tag.get_text(strip=True) if subtitulo_tag else ""

    # Número de notícias relacionadas linkadas na página
    relacionadas = soup_n.find_all("div", class_=lambda c: c and "news-last-description" in str(c))
    n_relacionadas = len(relacionadas)

    # Número de palavras do título
    n_palavras_titulo = len(title.split()) if title else 0

    fotografo, orgao_foto = parsear_credito(credito_foto)

    informacoes = {
        "url":              full_url,
        "titulo":           title,
        "subtitulo":        subtitulo,
        "descricao":        description,
        "temas":            temas,
        "imagem":           imagem,
        "fotografo":        fotografo,
        "orgao_foto":       orgao_foto,
        "data":             date_str,
        "hora_atualizacao": hora_atualizacao,
        "texto":            texto,
        "n_relacionadas":   n_relacionadas,
        "n_palavras_titulo": n_palavras_titulo,
    }

    file_path = data_dir / f"noticia_{i}.json"
    with file_path.open("w", encoding="utf-8") as f:
        json.dump(informacoes, f, ensure_ascii=False, indent=4)

    print(f"Arquivo salvo: {file_path.name}")
    time.sleep(random.uniform(0, 0.3))

# Executa em paralelo com 8 threads simultâneas
Parallel(n_jobs=8, prefer="threads")(
    delayed(processar_noticia)(i, link) for i, link in enumerate(links_noticias)
)

## Consolidando os dados coletados

Com todos os JSONs salvos, vamos carregar os arquivos e consolidar em um único `DataFrame` do pandas. Nessa etapa também derivamos colunas de tempo a partir do campo `data` (ano, mês, hora, turno) e calculamos métricas básicas como tamanho do texto e número de palavras do título.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import json
import pathlib

data_dir = pathlib.Path("data_minas")
registros = []

for json_file in data_dir.rglob("*.json"):
    try:
        with json_file.open("r", encoding="utf-8") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        print(f"Skipping corrupted file: {json_file}")
        continue

    date_str = data.get("data", "")
    date = None
    if date_str:
        try:
            date = datetime.fromisoformat(date_str)
        except ValueError:
            try:
                date = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S %z")
            except ValueError:
                try:
                    date = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
                except ValueError:
                    pass

    texto = data.get("texto", "")
    temas = data.get("temas", [])

    fotografo, orgao_foto = parsear_credito(data.get("credito_foto", ""))

    registros.append({
        "titulo":            data.get("titulo", ""),
        "subtitulo":         data.get("subtitulo", ""),
        "descricao":         data.get("descricao", ""),
        "temas":             temas,
        "data":              date,
        "ano_mes":           date.strftime("%Y-%m") if date else None,
        "ano":               date.year  if date else None,
        "mes":               date.month if date else None,
        "hora":              date.hour  if date else None,
        "turno":             ("Manhã" if date and date.hour < 12 else
                              "Tarde" if date and date.hour < 18 else
                              "Noite" if date else None),
        "n_temas":           len(temas),
        "tamanho_texto":     len(texto.split()),
        "n_palavras_titulo": data.get("n_palavras_titulo", 0),
        "n_relacionadas":    data.get("n_relacionadas", 0),
        "url":               data.get("url", ""),
    })

df = pd.DataFrame(registros)

print(f"Total de notícias carregadas: {len(df)}")
print(f"Com data:  {df['data'].notna().sum()}")
print(f"Com texto: {(df['tamanho_texto'] > 0).sum()}")
df.head()

In [ ]:
df.to_csv("data_minas/noticias_agencia_minas.csv", index=False, encoding="utf-8-sig")
print(f"CSV salvo com {len(df)} linhas!")

## Análise exploratória inicial

Com a base consolidada, fazemos uma primeira análise visual para entender o volume de publicações ao longo do tempo e quais temas dominam a cobertura esportiva do portal.

In [ ]:
# Volume de notícias por mês — mostra sazonalidade e evolução da cobertura
noticias_por_mes = df.dropna(subset=["ano_mes"]).groupby("ano_mes").size()

plt.figure(figsize=(14, 5))
plt.plot(noticias_por_mes.index, noticias_por_mes.values,
         marker="o", color="steelblue", linewidth=2)
plt.fill_between(noticias_por_mes.index, noticias_por_mes.values,
                 alpha=0.2, color="steelblue")
plt.xlabel("Mês")
plt.ylabel("Nº de Notícias")
plt.title("Volume de Notícias por Mês — Agência Minas Esportes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# Top 15 temas mais frequentes — mostra as prioridades editoriais do portal
df_temas = df.explode("temas").dropna(subset=["temas"])
tema_counts = df_temas["temas"].value_counts().head(15)

plt.figure(figsize=(12, 6))
bars = plt.bar(tema_counts.index, tema_counts.values,
               color=plt.cm.tab20.colors[:len(tema_counts)])
plt.xlabel("Tema")
plt.ylabel("Frequência")
plt.title("Top 15 Temas Mais Frequentes — Agência Minas")
plt.xticks(rotation=45, ha="right")
for bar, val in zip(bars, tema_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3, str(val), ha="center", fontsize=9)
plt.tight_layout()
plt.show()